In [15]:
import pandas as pd
import numpy as np
from pathlib import Path


In [23]:
df = pd.read_excel("asset_class_etf_data.xlsx", sheet_name = "excess returns", index_col=0, parse_dates=True)
assets = ["SPY", "VEA", "IEF", "HYG", "GLD", "USO"]
assets_and_btc = assets + ["BTC"]

df_in_sample = df[df.index <= "2021-12-31"]
df_out_of_sample = df[df.index >= "2022-01-01"]

In [27]:
# 5.1
def tangency_weights(ret, assets):
    mu = ret[assets].mean().values
    Sigma = ret[assets].cov().values
    Sigma_inv = np.linalg.inv(Sigma)
    w = Sigma_inv @ mu / (np.ones(len(assets)) @ Sigma_inv @ mu)
    return pd.Series(w, index=assets)

w_assets = tangency_weights(df_in_sample, assets)
w_assets_and_btc = tangency_weights(df_in_sample, assets_and_btc)

print(w_assets)
print(w_assets_and_btc)

SPY    1.974529
VEA   -0.827060
IEF    1.055115
HYG   -1.559205
GLD    0.402933
USO   -0.046312
dtype: float64
SPY    2.023425
VEA   -1.112864
IEF    1.008836
HYG   -1.302694
GLD    0.289217
USO   -0.056611
BTC    0.150690
dtype: float64


In [63]:
def portfolio_returns(ret_df, weights, assets):
    return ret_df[assets].values @ weights.values

r_out_of_sample = {
    "Tangency (ex-BTC)": portfolio_returns(df_out_of_sample, w_assets, assets),
    "Tangency (w/BTC)": portfolio_returns(df_out_of_sample, w_assets_and_btc, assets_and_btc),
    "Equal-Weight": df_out_of_sample[assets_and_btc].mean(axis=1).values,
    "60/40": 0.6 * df_out_of_sample["SPY"].values + 0.4 * df_out_of_sample["IEF"].values,
}

r_in_sample = {
    "Tangency (ex-BTC)": portfolio_returns(df_in_sample, w_assets, assets),
    "Tangency (w/BTC)": portfolio_returns(df_in_sample, w_assets_and_btc , assets_and_btc),
    "Equal-Weight": df_in_sample[assets_and_btc].mean(axis=1).values,
    "60/40": 0.6 * df_in_sample["SPY"].values + 0.4 * df_in_sample["IEF"].values,
}


In [65]:
def ann_stats(r_weekly):
    mu = r_weekly.mean() * 52
    vol = r_weekly.std(ddof=1) * np.sqrt(52)
    sr = mu / vol
    return mu, vol, sr

In [67]:
rows = []
for name, r in r_out_of_sample.items():
    mu, vol, sr = ann_stats(r)
    rows.append({"Portfolio":name, "Ann. Mean": mu, "Ann. Vol": vol, "Sharpe": sr})

    out_of_sample_table = pd.DataFrame(rows).set_index("Portfolio")
    print(out_of_sample_table.round(4).to_string())

                   Ann. Mean  Ann. Vol  Sharpe
Portfolio                                     
Tangency (ex-BTC)     0.1938    0.2031   0.954
                   Ann. Mean  Ann. Vol  Sharpe
Portfolio                                     
Tangency (ex-BTC)     0.1938    0.2031  0.9540
Tangency (w/BTC)      0.1769    0.2238  0.7905
                   Ann. Mean  Ann. Vol  Sharpe
Portfolio                                     
Tangency (ex-BTC)     0.1938    0.2031  0.9540
Tangency (w/BTC)      0.1769    0.2238  0.7905
Equal-Weight          0.1052    0.1239  0.8491
                   Ann. Mean  Ann. Vol  Sharpe
Portfolio                                     
Tangency (ex-BTC)     0.1938    0.2031  0.9540
Tangency (w/BTC)      0.1769    0.2238  0.7905
Equal-Weight          0.1052    0.1239  0.8491
60/40                 0.0499    0.1090  0.4575


In [69]:
# 5.2
rows_in_sample = []
for name, r in r_in_sample.items():
    mu, vol, sr = ann_stats(r)
    rows_in_sample.append({"Portfolio": name, "IS Mean": mu, "IS Vol": vol, "IS Sharpe": sr})

in_sample_table = pd.DataFrame(rows_in_sample).set_index("Portfolio")
comparison = in_sample_table[["IS Sharpe"]].join(out_of_sample_table[["Sharpe"]].rename(columns={"Sharpe": "OOS Sharpe"}))
comparison["Sharpe Decay (%)"] = (
    (comparison["IS Sharpe"] - comparison["OOS Sharpe"]) / comparison["IS Sharpe"] * 100
)

print(comparison.round(4).to_string())

                   IS Sharpe  OOS Sharpe  Sharpe Decay (%)
Portfolio                                                 
Tangency (ex-BTC)     1.8236      0.9540           47.6848
Tangency (w/BTC)      2.2253      0.7905           64.4744
Equal-Weight          1.2128      0.8491           29.9867
60/40                 1.0027      0.4575           54.3738


In [71]:
# We see that the ranking flips at the top: in sample tangency with BTC had the highest sharpe but out of sample it falls to second worst.
# Equal-weight is the most robust since it's Sharpe Decay is the lowest so this is likely the one that we would have actually wanted to hold. 

# 

In [ ]:
# 5.3
# This result shows the optimizer's fragility, it is sensitive to estimated means, has extreme unconstrained weights, 
# and there is Sharpe decay.  I wouldn't say it indicts the entire frameowkr but it shows the importance of doing more practical cases
# where you have weight constraints and shrinkage estimators.  